# ECC KV Cache — A100 Session
Run cells top to bottom. On runtime restart, re-run from Cell 1.

In [ ]:
# ── CELL 1: Setup (run every restart) ──────────────────────────────────────
import os, sys
os.chdir('/content/ecc-kv-cache')

# Pull latest fixes
!git pull origin main

# Clean stale build artifacts (prevents ABI mismatch between torch versions)
!rm -rf build/ custom_ecc_cuda*.so

# Build against currently installed torch (whatever Colab has)
!python setup.py build_ext --inplace 2>&1 | grep -v UserWarning | grep -v warnings.warn

# Verify
import custom_ecc_cuda
print('✓ Extension loaded')
print('✓ GPU:', __import__('torch').cuda.get_device_name(0))

In [ ]:
# ── CELL 2: CPU Unit Tests ──────────────────────────────────────────────────
!python -m pytest tests/unit/ -v --tb=short

In [ ]:
# ── CELL 3: HF Login (run once per VM session) ──────────────────────────────
from google.colab import userdata
from huggingface_hub import login
import os

# Change 'hf_token2' to your Colab secret name if different
token = userdata.get('hf_token2')
os.environ['HF_TOKEN'] = token
login(token=token)
print('✓ Logged in to HuggingFace')

In [ ]:
# ── CELL 4: Calibration (~10 min, run once per VM session) ──────────────────
MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'

!python scripts/run_calibration.py \
    --model {MODEL} \
    --n-samples 512 \
    --output calibration_config.json

import json
cfg = json.load(open('calibration_config.json'))
print(f'✓ Calibrated {len(cfg["layers"])} layers')

In [ ]:
# ── CELL 5: Smoke Test (~2 min) ─────────────────────────────────────────────
MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'

!python scripts/smoke_test.py \
    --model {MODEL} \
    --ctx-len 2000 \
    --n-tokens 50

print('✓ Smoke test passed — safe to run benchmarks')

In [ ]:
# ── CELL 6: VRAM Benchmark (~15 min) ────────────────────────────────────────
MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'
!mkdir -p results

!python benchmarks/vram_reduction.py \
    --model {MODEL} \
    --ctx-lengths 8000 32000 64000 128000 \
    --output results/vram_results.json

import json
r = json.load(open('results/vram_results.json'))
print('VRAM Results:')
for ctx, v in r.get('results', {}).items():
    print(f'  {int(ctx):>7,} tokens: FP16={v.get("fp16_peak_gb","OOM"):.1f}GB  ECC={v.get("ecc_peak_gb",0):.1f}GB  {v.get("reduction_x",0):.2f}x')

In [ ]:
# ── CELL 7: Throughput Benchmark (~15 min) ──────────────────────────────────
MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'

!python benchmarks/throughput.py \
    --model {MODEL} \
    --ctx-len 64000 \
    --n-tokens 200 \
    --output results/throughput.json

import json
r = json.load(open('results/throughput.json'))
print('Throughput Results:')
for method, v in r.get('results', {}).items():
    print(f'  {method}: {v.get("tps",0):.1f} tok/s')

In [ ]:
# ── CELL 8: NIAH Benchmark (~4.5 hours, crash-safe) ─────────────────────────
# Safe to interrupt and re-run — resumes from last saved trial automatically
MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'

!python benchmarks/niah_128k.py \
    --model {MODEL} \
    --ctx-lengths 8000 32000 64000 128000 \
    --depths 0.0 0.25 0.5 0.75 1.0 \
    --trials 100 \
    --methods fp16 int4_bnb int4_ecc \
    --output results/niah_results.json

In [ ]:
# ── CELL 9: Generate Comparison Table ───────────────────────────────────────
!python benchmarks/compare_baselines.py \
    --vram results/vram_results.json \
    --niah results/niah_results.json \
    --throughput results/throughput.json \
    --output results/comparison_table.md

print(open('results/comparison_table.md').read())

In [ ]:
# ── CELL 10: Push Results to GitHub ─────────────────────────────────────────
!git config user.email 'j-arndt@users.noreply.github.com'
!git config user.name 'j-arndt'
!git add results/ calibration_config.json
!git commit -m 'results: A100 80GB benchmark — VRAM, NIAH, throughput'
!git tag v0.1.0
!git push origin main --tags
print('✓ Results pushed to GitHub')